In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')
import torch
from transformer_lens import HookedTransformer
from dictionary_learning.mask_scae import SCAESuite
from interp_utils import prepare_streaming_dataset, generate_enhanced_viewer

# model_name = "roneneldan/TinyStories-33M"
model_name = "eleutherai/pythia-70m"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
expansion = 4
k = 30

# Initialize model
model = HookedTransformer.from_pretrained(model_name, device=device, dtype=torch.bfloat16)
tokenizer = model.tokenizer
# tinystories
# dataset_name = "roneneldan/TinyStories"
dataset_name = "kh4dien/pile-uncopyrighted-sample"
get_first_n = 10 # get first n features of each key

debug = True
if debug:
    max_length = 64
    batch_size = 16
    num_datapoints = 100
else:
    max_length = 64
    batch_size = 64
    num_datapoints = 5_000

/root/dictionary_learning/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded pretrained model eleutherai/pythia-70m into HookedTransformer


In [3]:
scae_repo_id = "jacobcd52/pythia-70m_c0_lr0.001_bs256_auxk0"
suite = SCAESuite.from_pretrained(
    repo_id=scae_repo_id,
    model=model,
    device=device,
)

In [4]:
model_save_name = scae_repo_id.split("/")[-1]
n_layers = suite.model.cfg.n_layers
keys = [f"attn_{i}" for i in range(n_layers)] + [f"mlp_{i}" for i in range(n_layers)]

In [10]:
from tqdm import tqdm
import os
import shutil
debug_interp = True
if debug_interp:
    # remove this folder
    folder_to_rm = "pythia-70m-deduped_scae_connections_20_llm_feature_viewer"
    if os.path.exists(folder_to_rm):
        shutil.rmtree(folder_to_rm)

    n_features = 15
    max_num_of_features_per_run = 10
    CHUNK_SIZE = 10
else:
    # n_features = suite.aes.attn_0.encoder.out_features
    n_features = 150
    max_num_of_features_per_run = 100
    CHUNK_SIZE = 100
num_runs = n_features // max_num_of_features_per_run + 1
# num_runs = 1
aes = {}
for module_name, module in suite.module_dict.items():
    aes[module_name] = module.ae


  0%|          | 0/2 [00:16<?, ?it/s]


KeyboardInterrupt: 

In [10]:
from IPython.display import display, HTML
num_feature_datapoints = 10 # how many examples/expert
for key in keys:
    key = "mlp_0"
    print(f"Key: {key}")
    features_for_this_key = features_to_save[key]
    for feature_idx, feature in enumerate(features_for_this_key):
        feature_activations = saved_feature_act_list[key][..., int(feature)]
        d_idx, seq_idx = get_feature_indices(feature_activations, k=num_feature_datapoints, setting="max")
        # uniform_indices = get_feature_indices(feature, feature_activations, k=num_feature_datapoints, setting="max")
        text_list, full_text, token_list, full_token_list, partial_activations, full_activations = get_feature_datapoints(d_idx, seq_idx, feature_activations, saved_token_list, tokenizer)
        html = tokens_and_activations_to_html(token_list, partial_activations, tokenizer)
        display(HTML(html))
        W_U = model.W_U
        final_ln = model.ln_final
        feature_decoder = suite.aes[key].decoder.weight[:, int(feature)]
        logit_lens = final_ln(feature_decoder) @ W_U
        top_val, top_ind = torch.topk(logit_lens, k=10, dim=-1)
        bot_val, bot_ind = torch.topk(logit_lens, k=10, dim=-1, largest=False)
        logit_lens_html = create_logit_lens_html(top_ind, top_val, bot_ind, bot_val, tokenizer)
        display(HTML(logit_lens_html))
        if(feature_idx > 10):
            break # to avoid too many examples
    break

Key: mlp_0


Top Token,Value,Bottom Token,Value
_suddenly,15.125,nd,-11.130
_we,12.100,Four,-10.588
_you,12.096,_Mostly,-10.065
_Humans,12.069,_kale,-9.926
_come,11.732,_amounts,-9.913
_finally,11.604,_then,-9.489
_exhaustion,11.275,_rate,-9.448
_Enter,11.127,mos,-9.355
_introduce,11.056,_gunshots,-9.026
_magically,10.942,_Mid,-9.022


Top Token,Value,Bottom Token,Value
_cat,13.659,ling,-12.546
fly,10.926,ink,-12.477
_Zig,10.849,ering,-12.035
_fly,10.792,_fr,-11.838
jo,10.710,out,-11.718
_Ana,10.554,aming,-11.160
_audition,10.497,et,-11.157
_musician,10.393,er,-11.062
_Rico,10.272,hole,-10.951
_sk,10.111,_manners,-10.778


Top Token,Value,Bottom Token,Value
urry,14.061,_gratification,-13.370
_oily,13.920,_motivation,-11.781
_pink,13.646,ms,-11.226
_fluffy,12.550,_ampl,-10.972
bing,11.556,_dominated,-10.623
_cute,11.520,beh,-9.798
_but,11.216,Things,-9.776
_graceful,11.093,azard,-9.638
_mush,11.011,_securing,-9.638
_polite,11.005,Sher,-9.595


Top Token,Value,Bottom Token,Value
_director,12.904,_his,-16.349
_happily,12.376,_her,-13.949
_Tee,11.914,ged,-11.924
_Oct,11.100,His,-11.461
"_""",10.769,pt,-11.393
_Ma,10.410,Her,-11.353
_Age,10.319,_him,-11.288
Cle,10.177,bite,-11.202
_Whe,10.151,ared,-11.145
_Arrows,10.089,_thinner,-10.879


...

Top Token,Value,Bottom Token,Value
y,15.621,_blaze,-10.512
pped,14.033,_proposal,-10.221
c,13.492,_setback,-10.003
_announcing,12.819,_fend,-9.921
is,11.922,_Among,-9.441
lessly,11.778,_miracle,-9.271
ler,11.700,_differences,-9.174
fl,11.658,_particular,-9.093
v,11.359,Sony,-9.071
mail,11.253,Both,-8.806


Top Token,Value,Bottom Token,Value
_that,13.333,_disgust,-11.818
_little,12.382,abb,-11.142
_place,12.328,Av,-10.045
_well,11.687,keys,-9.939
_luck,11.452,Design,-9.813
_the,11.230,_murm,-9.658
_game,11.106,_const,-9.170
_games,10.935,_unravel,-9.109
_her,10.830,_fails,-9.030
_good,10.800,_pleaded,-8.964


Top Token,Value,Bottom Token,Value
_upstream,11.564,upp,-13.161
_downstream,10.777,ep,-11.499
_emotional,10.689,_Play,-11.450
_determined,10.339,ci,-11.397
_fertilizer,10.181,room,-11.285
_American,9.923,_fists,-11.177
tree,9.862,_play,-10.445
_past,9.818,AY,-10.324
"!'""",9.759,_doors,-10.248
_Often,9.668,ala,-10.090


Top Token,Value,Bottom Token,Value
rm,11.075,_grips,-13.360
_orphan,10.754,Dis,-11.128
aze,10.278,ner,-10.726
_Circus,9.826,sed,-10.308
_disapp,9.709,_deserve,-10.269
_Broken,9.681,_cores,-10.212
_result,9.516,_thirteen,-10.089
_from,9.487,aying,-10.069
_combo,9.432,d,-10.025
_threshold,9.408,_placing,-9.982


Top Token,Value,Bottom Token,Value
_work,13.825,_sadd,-13.262
_soak,13.709,imp,-10.887
_dry,13.502,die,-10.859
_action,12.719,_rushes,-10.473
_places,11.298,_receiving,-10.078
_inspire,11.263,_anew,-9.744
_haul,10.870,tw,-9.737
_drown,10.667,ents,-9.576
_lure,10.626,_Homer,-9.501
asty,10.540,heard,-9.488


In [11]:
key, feature

('mlp_0', '11')